In [1]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

In [2]:
df=pd.read_csv('train.csv')

In [3]:
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
707,708,1,1,"Calderhead, Mr. Edward Pennington",male,42.0,0,0,PC 17476,26.2875,E24,S
869,870,1,3,"Johnson, Master. Harold Theodor",male,4.0,1,1,347742,11.1333,NaN,S
242,243,0,2,"Coleridge, Mr. Reginald Charles",male,29.0,0,0,W./C. 14263,10.5000,NaN,S
31,32,1,1,"Spencer, Mrs. William Augustus (Marie Eugenie)",female,NaN,1,0,PC 17569,146.5208,B78,C
519,520,0,3,"Pavlovic, Mr. Stefo",male,32.0,0,0,349242,7.8958,NaN,S


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [5]:
# Fill missing values in Age with the median to handle missing data without introducing outliers
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')
df['Age'] = imputer.fit_transform(df[['Age']])

In [6]:
# Drop the few rows where 'Embarked' is missing to keep clean target data
df = df.dropna(subset=['Embarked'])

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 889 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  889 non-null    int64  
 1   Survived     889 non-null    int64  
 2   Pclass       889 non-null    int64  
 3   Name         889 non-null    object 
 4   Sex          889 non-null    object 
 5   Age          889 non-null    float64
 6   SibSp        889 non-null    int64  
 7   Parch        889 non-null    int64  
 8   Ticket       889 non-null    object 
 9   Fare         889 non-null    float64
 10  Cabin        202 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 90.3+ KB


In [17]:
# Encode categorical variables into numbers so the ML models can process them
oe = OrdinalEncoder()
df['Sex'] = oe.fit_transform(df[['Sex']])  # Female -> 0, Male -> 1

In [9]:
ohe=OneHotEncoder(sparse_output=False,drop='first')
embarked_encoded=ohe.fit_transform(df[['Embarked']])
encoded_cols=ohe.get_feature_names_out(['Embarked'])
df_embarked=pd.DataFrame(embarked_encoded,columns=encoded_cols,index=df.index)
df=pd.concat([df,df_embarked],axis=1)
df=df.drop(columns=['Embarked'])

In [10]:
X=df[['Pclass','Sex','Fare','Age','Embarked_Q','Embarked_S']]

In [11]:
y=df['Survived']

In [19]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

# TRAIN-TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# MODEL TRAINING (Decision Tree Classifier)
clf=DecisionTreeClassifier(max_depth=4, random_state=42)
clf.fit(X_train,y_train)
y_pred=clf.predict(X_test)


In [18]:
# MODEL TRAINING (LogisticRegression)
from sklearn.linear_model import LogisticRegression
lr_clf = LogisticRegression(max_iter=1000) # max_iter ensures the model has enough time to find the best fit
lr_clf.fit(X_train, y_train)

y_pred_lr = lr_clf.predict(X_test)

In [20]:
# EVALUATION
accuracy = clf.score(X_test, y_test)
print("DT",accuracy)
accuracy = lr_clf.score(X_test, y_test)
print("LR",accuracy)

DT 0.8089887640449438
LR 0.7752808988764045


In [21]:
# EVALUATION
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.83      0.86      0.85       109
           1       0.77      0.72      0.75        69

    accuracy                           0.81       178
   macro avg       0.80      0.79      0.80       178
weighted avg       0.81      0.81      0.81       178



In [22]:
# MODEL TRAINING (Random Forest Classifier)
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf.fit(X_train, y_train)
print("RF Accuracy:", rf_clf.score(X_test, y_test))

RF Accuracy: 0.8146067415730337
